# BESS Layout Optimization Tool

This notebook implements a layout optimization engine for Battery Energy Storage Systems (BESS).

## Objectives

- Define an arbitrary site using a polygon
- Place different types of equipment:
  - BESS containers (20 ft)
  - MVS / PCS containers (20 ft)
- Define:
  - Equipment footprint (2D)
  - Clearance distances (overlapping allowed)
- Configure maximum number of BESS per MVS
- Optimize layout to maximize battery count under constraints

## Key Assumptions

- Rectangular equipment footprints
- Clearance zones are allowed to overlap
- Equipment footprints cannot overlap
- Each BESS must be assigned to a MVS

In [24]:
# Install required packages (run once)

!pip install shapely matplotlib numpy

# Environment Check

This section ensures that all required libraries are correctly installed.

If this cell runs successfully, the environment is ready to execute the layout optimization workflow.

In [25]:
# Quick sanity check

from shapely.geometry import Polygon

poly = Polygon([(0, 0), (10, 0), (10, 10), (0, 10)])

print("✅ Shapely working correctly:", poly.area == 100)

✅ Shapely working correctly: True


# Configuration

In this section, the main parameters of the layout problem are defined.

## Site Definition
- The site is defined as a polygon using a list of vertices (X, Y)

## Zone Types

Two types of zones are defined:

### 1. Non-buildable areas
- Equipment cannot be placed
- Can still be used for:
  - cable routing
  - access paths
  - fire lanes
- These areas block placement but do not reduce total usable layout space

### 2. Restricted areas
- Completely excluded from the site
- No usage allowed
- These areas are subtracted from the site geometry

## Equipment Types
- BESS containers
- MVS / PCS containers

Each equipment type includes:
- Footprint (width and height)
- Clearance (allowed to overlap with other clearance zones)

## System Constraints
- Maximum number of BESS units per MVS
- Grid resolution for placement candidates

This structure allows modeling realistic BESS layouts with engineering constraints.

In [33]:

blue_zone = [
    (15.4, 0),
    (21.9, 0),
    (21.9, 90.4),
    (15.4, 90.4)
]

blue_diagonal = [
    (21.9, 0),
    (35, 0),
    (28, -10),
    (15.4, 0)
]

orange_zone_left = [
    (0, 0),
    (8, 0),
    (12, 85),
    (0, 90)
]



In [ ]:
# =========================================================
# KEY INPUTS — edit here before running
# =========================================================

# --- Equipment footprint dimensions (metres) ---
BESS_WIDTH  = 6.06
BESS_HEIGHT = 2.44

MVS_WIDTH   = 6.06
MVS_HEIGHT  = 2.44

# --- Clearance distances (metres) — asymmetric per side ---
# Each side can have a different value.
#   front = facing the access aisle (+ Y direction)
#   back  = rear of the container   (- Y direction)
#   left  = left side               (- X direction)
#   right = right side              (+ X direction)
#
# Clearances are shown in the plot but are NOT hard exclusions:
# clearance zones CAN overlap with each other — only footprints cannot.

BESS_CLEARANCE = {
    "front": 2.0,   # main O&M aisle side
    "back":  1.0,
    "left":  1.0,
    "right": 1.0,
}

MVS_CLEARANCE = {
    "front": 3.0,   # larger aisle for cable and O&M access
    "back":  1.5,
    "left":  1.5,
    "right": 1.5,
}

# --- System constraints ---
MAX_BESS_PER_MVS = 4      # maximum BESS units a single MVS can serve
MIN_MVS_SPACING  = 15.0   # minimum centre-to-centre distance between MVS units (m)

# --- Layout resolution ---
GRID_RESOLUTION = 2.5     # candidate grid step (m) — smaller = denser search, slower
SETBACK         = 0       # inward boundary offset (m)


In [ ]:
# =========================================================
# CONFIGURATION  (references KEY INPUTS above)
# =========================================================

CONFIG = {

    # ---------------------------
    # SITE DEFINITION (polygon)
    # ---------------------------
    "site_vertices": [
        (0, 0),
        (53.3, 0),
        (53.3, 90.4),
        (21.4, 90.4),
        (10, 10),
        (0, 20)
    ],

    "setback": SETBACK,

    # ---------------------------
    # ZONES
    # ---------------------------
    "zones": {
        # Cannot place equipment, but usable space (routing, fire lanes, etc.)
        "non_buildable": [
            blue_zone,
            blue_diagonal,
            orange_zone_left
        ],
        # Completely excluded from site geometry
        "restricted": []
    },

    # ---------------------------
    # EQUIPMENT TYPES
    # ---------------------------
    "equipment": {

        "BESS": {
            "width":     BESS_WIDTH,
            "height":    BESS_HEIGHT,
            "clearance": BESS_CLEARANCE
        },

        "MVS": {
            "width":     MVS_WIDTH,
            "height":    MVS_HEIGHT,
            "clearance": MVS_CLEARANCE
        }
    },

    # ---------------------------
    # SYSTEM CONSTRAINTS
    # ---------------------------
    "max_bess_per_mvs": MAX_BESS_PER_MVS,
    "min_mvs_spacing":  MIN_MVS_SPACING,
    "grid_resolution":  GRID_RESOLUTION,
}


# Geometry and Site Preparation

This section defines how the site geometry is prepared before placing equipment.

## Key Steps

1. Create the base site polygon from vertices
2. Apply setback (inward buffer)
3. Subtract restricted areas from the site
4. Keep non-buildable zones as internal constraints
5. Generate candidate placement positions using a grid

## Important Behavior

- Restricted areas → removed from the usable site
- Non-buildable areas → remain inside the site but block placement
- Clearance between equipment is allowed to overlap

The combination of these rules allows realistic modeling of:
- access roads
- fire lanes
- no-build zones
- environmental restrictions

In [27]:
from shapely.geometry import Polygon
import numpy as np

def create_site(vertices):
    site = Polygon(vertices)

    if not site.is_valid:
        raise ValueError("Invalid polygon: check vertex order")

    return site


def prepare_site(site, config):

    # 1. Apply setback
    usable_site = site.buffer(-config["setback"])

    # 2. Remove restricted zones
    for zone in config["zones"]["restricted"]:
        zone_poly = Polygon(zone)
        usable_site = usable_site.difference(zone_poly)

    # 3. Keep non-buildable zones (for placement checks later)
    non_buildable_polys = [
        Polygon(z) for z in config["zones"]["non_buildable"]
    ]

    return usable_site, non_buildable_polys


def create_candidate_grid(site, resolution):
    minx, miny, maxx, maxy = site.bounds

    xs = np.arange(minx, maxx, resolution)
    ys = np.arange(miny, maxy, resolution)

    return [(x, y) for x in xs for y in ys]

# Equipment Placement Logic

This section defines how equipment is placed within the site.

## Placement Rules

### 1. Footprint constraints
- Equipment footprints must:
  - Be fully inside the usable site
  - Not intersect with other equipment
  - Not overlap with non-buildable zones

### 2. Clearance concept
- Each equipment has a clearance distance
- Clearance areas:
  - Can overlap with other clearances
  - Represent safety, fire, or O&M spacing
- Clearance is NOT treated as a hard exclusion

### 3. Placement Strategy

The placement is done sequentially:

1. Place MVS units first
2. Assign BESS units to MVS based on proximity
3. Ensure each MVS does not exceed maximum BESS capacity

## Objective

Maximize the number of BESS units while ensuring:
- Proper assignment to MVS
- Valid spatial layout


In [ ]:
from shapely.geometry import box

def create_equipment_polygon(x, y, width, height):
    return box(x, y, x + width, y + height)


def create_clearance_polygon(footprint, clearance):
    """
    Build an asymmetric clearance rectangle around a footprint.
    clearance dict keys: front (+Y), back (-Y), left (-X), right (+X).
    """
    minx, miny, maxx, maxy = footprint.bounds
    return box(
        minx - clearance["left"],
        miny - clearance["back"],
        maxx + clearance["right"],
        maxy + clearance["front"],
    )


def is_valid_placement(candidate, site, non_buildable, placed):

    # Must be fully inside usable site
    if not site.contains(candidate):
        return False

    # Must not intersect non-buildable zones
    for zone in non_buildable:
        if candidate.intersects(zone):
            return False

    # Must not overlap other equipment footprints
    for obj in placed:
        if candidate.intersects(obj["footprint"]):
            return False

    return True


# MVS and BESS Placement Strategy

This section defines the logic used to allocate equipment across the site.

## MVS Placement

- MVS units are placed first
- They act as anchors for BESS allocation
- Placement is based on valid grid positions

## BESS Placement

- BESS units are placed after MVS
- Each BESS unit is assigned to the nearest MVS
- Assignment is constrained by:
  - Maximum BESS per MVS

## Optimization Philosophy

- Prioritize filling MVS to maximum capacity
- Use proximity (distance) as assignment metric
- Avoid scattered layouts where possible

This approach approximates real BESS design methodology where:
- Power blocks are centered around PCS/MVS
- Batteries are clustered around conversion units

In [ ]:
def place_mvs(site, non_buildable, grid, config):
    """
    Place MVS units on the grid with a minimum centre-to-centre spacing
    so each MVS has territory to host a full BESS cluster.
    """
    mvs_list = []
    placed = []
    eq = config["equipment"]["MVS"]
    min_spacing = config.get("min_mvs_spacing", 0)

    for (x, y) in grid:
        poly = create_equipment_polygon(x, y, eq["width"], eq["height"])

        if not is_valid_placement(poly, site, non_buildable, placed):
            continue

        # Reject if too close to an already-placed MVS
        cx, cy = poly.centroid.x, poly.centroid.y
        too_close = any(
            (cx - m["footprint"].centroid.x) ** 2 + (cy - m["footprint"].centroid.y) ** 2
            < min_spacing ** 2
            for m in mvs_list
        )
        if too_close:
            continue

        obj = {
            "type": "MVS",
            "footprint": poly,
            "assigned_bess": []
        }
        mvs_list.append(obj)
        placed.append(obj)

    return mvs_list, placed


def place_bess(site, non_buildable, grid, config, mvs_list, placed):
    """
    MVS-centric placement: for each MVS, sort all grid candidates by proximity
    and fill that MVS to capacity before moving to the next one.
    Produces compact clusters instead of a scattered global assignment.
    """
    bess_list = []
    eq = config["equipment"]["BESS"]
    max_ratio = config["max_bess_per_mvs"]

    for mvs in mvs_list:
        cx = mvs["footprint"].centroid.x
        cy = mvs["footprint"].centroid.y

        # Nearest-first search relative to this MVS
        sorted_candidates = sorted(
            grid,
            key=lambda pt: (pt[0] - cx) ** 2 + (pt[1] - cy) ** 2
        )

        for (x, y) in sorted_candidates:
            if len(mvs["assigned_bess"]) >= max_ratio:
                break  # MVS is full — stop searching

            poly = create_equipment_polygon(x, y, eq["width"], eq["height"])

            if not is_valid_placement(poly, site, non_buildable, placed):
                continue

            obj = {
                "type": "BESS",
                "footprint": poly,
                "mvs": mvs  # reference enables connection lines in visualisation
            }
            bess_list.append(obj)
            placed.append(obj)
            mvs["assigned_bess"].append(obj)

    return bess_list


# Optimization Execution

This section combines all previous components into a single workflow.

## Process Overview

1. Create base site geometry
2. Apply setback and zone constraints
3. Generate candidate grid points
4. Place MVS units across the site
5. Allocate BESS units to MVS based on proximity and capacity
6. Compute final metrics

## Output

- Total number of MVS units
- Total number of BESS units
- Average BESS per MVS ratio

This step produces the final layout data used for visualization.


In [ ]:
def run_optimization(config):

    # Step 1: Create base site
    site_raw = create_site(config["site_vertices"])

    # Step 2: Apply constraints (setback + zones)
    site, non_buildable = prepare_site(site_raw, config)

    # Step 3: Generate candidate grid
    grid = create_candidate_grid(site, config["grid_resolution"])

    # Step 4: Place MVS (spaced anchors)
    mvs_list, placed = place_mvs(site, non_buildable, grid, config)

    # Step 5: Place BESS in clusters around each MVS
    bess_list = place_bess(site, non_buildable, grid, config, mvs_list, placed)

    # Step 6: Report
    total_mvs  = len(mvs_list)
    total_bess = len(bess_list)
    max_cap    = config["max_bess_per_mvs"]
    avg_ratio  = total_bess / total_mvs if total_mvs > 0 else 0
    full_mvs   = sum(1 for m in mvs_list if len(m["assigned_bess"]) == max_cap)

    print("\n===== RESULTS =====")
    print(f"MVS units placed   : {total_mvs}")
    print(f"BESS units placed  : {total_bess}")
    print(f"Max BESS per MVS   : {max_cap}")
    print(f"Avg BESS per MVS   : {avg_ratio:.2f}")
    print(f"Fully saturated MVS: {full_mvs} / {total_mvs}")
    if total_mvs:
        print(f"Saturation rate    : {100 * total_bess / (total_mvs * max_cap):.1f}%")

    return site, non_buildable, mvs_list, bess_list


# Visualization

This section generates a 2D layout representation of the optimized site.

## Color Legend

- Black → Site boundary
- Yellow → Non-buildable zones
- Red → MVS containers
- Blue → BESS containers

## Purpose

The visualization allows:
- Quick validation of layout feasibility
- Identification of unused areas
- Communication with stakeholders (engineering / commercial)

This output is critical for early-stage design and proposal development.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

def plot_layout(site, non_buildable, mvs_list, bess_list, config):

    bess_cl = config["equipment"]["BESS"]["clearance"]
    mvs_cl  = config["equipment"]["MVS"]["clearance"]

    fig, ax = plt.subplots(figsize=(12, 10))

    # --- Site boundary ---
    if not site.is_empty:
        x, y = site.exterior.xy
        ax.plot(x, y, color="black", linewidth=2)

    # --- Non-buildable zones ---
    for zone in non_buildable:
        if not zone.is_empty:
            x, y = zone.exterior.xy
            ax.fill(x, y, color="gold", alpha=0.45)
            ax.plot(x, y, color="goldenrod", linewidth=1)

    # --- BESS clearance zones (light blue, drawn first) ---
    for bess in bess_list:
        cl_poly = create_clearance_polygon(bess["footprint"], bess_cl)
        x, y = cl_poly.exterior.xy
        ax.fill(x, y, color="lightsteelblue", alpha=0.30, zorder=1)
        ax.plot(x, y, color="steelblue", linewidth=0.5, linestyle="--", alpha=0.6, zorder=1)

    # --- MVS clearance zones (light salmon) ---
    for mvs in mvs_list:
        cl_poly = create_clearance_polygon(mvs["footprint"], mvs_cl)
        x, y = cl_poly.exterior.xy
        ax.fill(x, y, color="lightsalmon", alpha=0.35, zorder=1)
        ax.plot(x, y, color="tomato", linewidth=0.5, linestyle="--", alpha=0.6, zorder=1)

    # --- Connection lines: BESS centroid → assigned MVS centroid ---
    for bess in bess_list:
        if "mvs" not in bess:
            continue
        ax.plot(
            [bess["footprint"].centroid.x, bess["mvs"]["footprint"].centroid.x],
            [bess["footprint"].centroid.y, bess["mvs"]["footprint"].centroid.y],
            color="gray", linewidth=0.6, alpha=0.5, zorder=2
        )

    # --- BESS footprints ---
    for bess in bess_list:
        x, y = bess["footprint"].exterior.xy
        ax.fill(x, y, color="steelblue", alpha=0.85, zorder=3)
        ax.plot(x, y, color="navy", linewidth=0.5, zorder=3)

    # --- MVS footprints (on top) ---
    for mvs in mvs_list:
        x, y = mvs["footprint"].exterior.xy
        ax.fill(x, y, color="crimson", alpha=0.90, zorder=4)
        ax.plot(x, y, color="darkred", linewidth=0.8, zorder=4)
        # Label each MVS with its assigned BESS count
        ax.text(
            mvs["footprint"].centroid.x, mvs["footprint"].centroid.y,
            str(len(mvs["assigned_bess"])),
            ha="center", va="center", fontsize=6,
            color="white", fontweight="bold", zorder=5
        )

    # --- Legend ---
    handles = [
        mpatches.Patch(color="steelblue",      alpha=0.85, label=f"BESS footprint ({len(bess_list)} units)"),
        mpatches.Patch(color="lightsteelblue", alpha=0.50, label="BESS clearance"),
        mpatches.Patch(color="crimson",        alpha=0.90, label=f"MVS footprint ({len(mvs_list)} units)"),
        mpatches.Patch(color="lightsalmon",    alpha=0.50, label="MVS clearance"),
        mpatches.Patch(color="gold",           alpha=0.45, label="Non-buildable zone"),
    ]
    ax.legend(handles=handles, loc="upper right", fontsize=8)

    ax.set_aspect("equal")
    ax.set_title("BESS Layout Optimization", fontsize=14, fontweight="bold")
    ax.set_xlabel("X (m)")
    ax.set_ylabel("Y (m)")
    ax.grid(True, linestyle="--", alpha=0.3)
    plt.tight_layout()
    plt.show()


In [ ]:
site, non_buildable, mvs, bess = run_optimization(CONFIG)
plot_layout(site, non_buildable, mvs, bess, CONFIG)
